In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras import layers

In [3]:
model_full = load_model("/content/drive/MyDrive/b/c/modelo_categoria_full.keras")
model_reducido = load_model("/content/drive/MyDrive/b/c/modelo_categoria_reducido.keras")

with open("/content/drive/MyDrive/b/c/artefactos_categoria.pkl", "rb") as f:
    artefactos = pickle.load(f)

label_encoder = artefactos['label_encoder']
vocabulario = artefactos['config_vectorizador']['vocabulario']

print(f"Clases categoria_principal: {list(label_encoder.classes_)}")

Clases categoria_principal: ['Alimentacion', 'Entretenimiento', 'Finanzas', 'Hogar', 'Salud', 'Transporte']


In [4]:
max_tokens = 5000
sequence_length = 5

vectorize_layer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=sequence_length
)

vectorize_layer.set_vocabulary(vocabulario)

print(f"Vocabulario cargado: {len(vocabulario)} tokens")

Vocabulario cargado: 136 tokens


In [5]:
def normalizar_esencial(valor):
    """Convierte 'esencial' (si/no, true/false, 1/0) a float 0/1."""
    mapa = {'si': 1, 'sí': 1, 'true': 1, '1': 1, 'no': 0, 'false': 0, '0': 0}
    if isinstance(valor, str):
        return float(mapa.get(valor.strip().lower(), 0))
    return float(valor)

In [6]:
def predecir_transaccion_full(nombre_tienda, subcategoria, esencial):
    """
    Modelo completo: usa nombre_tienda + subcategoria + esencial.
    Devuelve la categoria_principal predicha con su confianza.
    """
    input_nombre = tf.constant([nombre_tienda], dtype=tf.string)
    input_subcategoria = tf.constant([subcategoria], dtype=tf.string)
    input_esencial = tf.constant([[normalizar_esencial(esencial)]], dtype=tf.float32)

    pred = model_full.predict(
        {'input_nombre': input_nombre, 'input_subcategoria': input_subcategoria, 'input_esencial': input_esencial},
        verbose=0
    )

    idx = np.argmax(pred[0])
    categoria = label_encoder.inverse_transform([idx])[0]
    confianza = float(pred[0][idx])

    return {'categoria_principal': categoria, 'confianza': confianza}

In [7]:
def predecir_transaccion_reducido(subcategoria, esencial):
    """
    Modelo reducido: usa solo subcategoria + esencial (sin nombre_tienda).
    Devuelve la categoria_principal predicha con su confianza.
    """
    input_subcategoria = tf.constant([subcategoria], dtype=tf.string)
    input_esencial = tf.constant([[normalizar_esencial(esencial)]], dtype=tf.float32)

    pred = model_reducido.predict(
        {'input_subcategoria': input_subcategoria, 'input_esencial': input_esencial},
        verbose=0
    )

    idx = np.argmax(pred[0])
    categoria = label_encoder.inverse_transform([idx])[0]
    confianza = float(pred[0][idx])

    return {'categoria_principal': categoria, 'confianza': confianza}

In [10]:
resultado_full = predecir_transaccion_full("Don Julio Parrilla", "restaurante", "no")
print(f"[Full] Categoría: {resultado_full['categoria_principal']} ({resultado_full['confianza']:.2%})")

resultado_reducido = predecir_transaccion_reducido("restaurante", "no")
print(f"[Reducido] Categoría: {resultado_reducido['categoria_principal']} ({resultado_reducido['confianza']:.2%})")

[Full] Categoría: Alimentacion (99.92%)
[Reducido] Categoría: Alimentacion (100.00%)
